# KADMON — Exporter les centroïdes QuickBundles en NumPy

Ce notebook charge un bundle source et un bundle cible, les compresse séparément avec QuickBundles, puis exporte **uniquement leurs centroïdes** dans deux fichiers `.npy`. Les poids des clusters ne sont pas exportés.

In [ ]:
from pathlib import Path
import sys

# Fonctionne lorsque Jupyter est lancé depuis la racine ou depuis notebooks/.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "kadmon").is_dir() else CURRENT_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"
SOURCE_PATH = BUNDLES_DIR / "103818/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m.trk"
TARGET_PATH = BUNDLES_DIR / "433839/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m.trk"

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "centroids_export"
SOURCE_OUTPUT = OUTPUT_DIR / "source_centroids.npy"
TARGET_OUTPUT = OUTPUT_DIR / "target_centroids.npy"

N_POINTS = 12
QUICKBUNDLES_THRESHOLD_MM = 7.0

## Charger et compresser les bundles

In [ ]:
from kadmon.compression import compress_quickbundles
from kadmon.io import load_bundle

for label, path in (("source", SOURCE_PATH), ("cible", TARGET_PATH)):
    if not path.is_file():
        raise FileNotFoundError(f"Bundle {label} introuvable : {path}")

source_bundle = load_bundle(SOURCE_PATH, n_points=N_POINTS)
target_bundle = load_bundle(TARGET_PATH, n_points=N_POINTS)

source_centroids, _ = compress_quickbundles(
    source_bundle, threshold=QUICKBUNDLES_THRESHOLD_MM
)
target_centroids, _ = compress_quickbundles(
    target_bundle, threshold=QUICKBUNDLES_THRESHOLD_MM
)

print(f"Source : {len(source_bundle)} streamlines → {len(source_centroids)} centroïdes")
print(f"Cible  : {len(target_bundle)} streamlines → {len(target_centroids)} centroïdes")
print(f"Forme source : {source_centroids.shape}")
print(f"Forme cible  : {target_centroids.shape}")

## Exporter uniquement les centroïdes

In [ ]:
import numpy as np

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(SOURCE_OUTPUT, source_centroids)
np.save(TARGET_OUTPUT, target_centroids)

print(f"Source exportée : {SOURCE_OUTPUT}")
print(f"Cible exportée  : {TARGET_OUTPUT}")

## Vérifier les fichiers exportés

In [ ]:
saved_source = np.load(SOURCE_OUTPUT)
saved_target = np.load(TARGET_OUTPUT)

assert np.array_equal(saved_source, source_centroids)
assert np.array_equal(saved_target, target_centroids)
print("Export vérifié.")
print(f"{SOURCE_OUTPUT.name}: shape={saved_source.shape}, dtype={saved_source.dtype}")
print(f"{TARGET_OUTPUT.name}: shape={saved_target.shape}, dtype={saved_target.dtype}")